## Topic: A Complete RAG Pipeline as a Sequential Graph

In [ ]:
"""
Example 3: Sequential RAG Pipeline
───────────────────────────────────
Flow: START → retrieve → grade → generate → format → END

This is a REAL-WORLD pattern used in production RAG systems.
Each step feeds into the next. No branching (yet — that's CRAG).
"""

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Setup LLM and vector store
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Sample knowledge base
docs = [
    Document(page_content="LangGraph is a framework for building stateful AI agents.",
             metadata={"source": "docs/intro.md"}),
    Document(page_content="LangGraph uses graphs with nodes, edges, and state.",
             metadata={"source": "docs/concepts.md"}),
    Document(page_content="LangGraph supports sequential, parallel, and conditional workflows.",
             metadata={"source": "docs/workflows.md"}),
    Document(page_content="Checkpoints in LangGraph enable persistence and memory.",
             metadata={"source": "docs/memory.md"}),
]
vectorstore = FAISS.from_documents(docs, OpenAIEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})


# 1. STATE
class RAGState(TypedDict):
    question: str
    documents: list[str]
    relevance_score: float
    answer: str
    formatted_output: str


# 2. NODES
def retrieve_node(state: RAGState) -> dict:
    """Step 1: Retrieve relevant documents."""
    print("  [1/4] Retrieving documents...")
    results = retriever.invoke(state["question"])
    doc_texts = [doc.page_content for doc in results]
    print(f"         Found {len(doc_texts)} documents")
    return {"documents": doc_texts}


def grade_node(state: RAGState) -> dict:
    """Step 2: Grade document relevance."""
    print("  [2/4] Grading relevance...")
    # Simple grading: check if question keywords appear in documents
    question_words = set(state["question"].lower().split())
    doc_words = set(" ".join(state["documents"]).lower().split())
    overlap = len(question_words & doc_words) / max(len(question_words), 1)
    score = min(overlap * 2, 1.0)  # Normalize to 0-1
    print(f"         Relevance score: {score:.2f}")
    return {"relevance_score": score}


def generate_node(state: RAGState) -> dict:
    """Step 3: Generate answer from documents."""
    print("  [3/4] Generating answer...")
    context = "\n".join(state["documents"])
    response = llm.invoke(
        f"Answer the question using ONLY this context:\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {state['question']}\n\n"
        f"Answer:"
    )
    return {"answer": response.content}


def format_node(state: RAGState) -> dict:
    """Step 4: Format the final output."""
    print("  [4/4] Formatting output...")
    formatted = (
        f"📋 Question: {state['question']}\n"
        f"📊 Relevance: {state['relevance_score']:.0%}\n"
        f"💬 Answer: {state['answer']}\n"
        f"📚 Sources: {len(state['documents'])} documents"
    )
    return {"formatted_output": formatted}


# 3. BUILD SEQUENTIAL GRAPH
graph = StateGraph(RAGState)

graph.add_node("retrieve", retrieve_node)
graph.add_node("grade", grade_node)
graph.add_node("generate", generate_node)
graph.add_node("format", format_node)

# Sequential edges
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "grade")
graph.add_edge("grade", "generate")
graph.add_edge("generate", "format")
graph.add_edge("format", END)

# 4. COMPILE & RUN
app = graph.compile()

print("🚀 Running Sequential RAG Pipeline\n")

result = app.invoke({
    "question": "What is LangGraph and what workflows does it support?",
    "documents": [],
    "relevance_score": 0.0,
    "answer": "",
    "formatted_output": ""
})

print(f"\n✅ Final Output:\n{result['formatted_output']}")